In [0]:
%run "../includes/configuration"

In [0]:
movies_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movies")
).select("movie_id", "budget", "revenue", "year_release_date")


In [0]:
movies_genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/movie_genre")
).select("genre_id", "movie_id")

In [0]:
genres_df = (
    spark.read
        .format("parquet")
        .load(f"{silver_folder_path}/genres")
).select("genre_id", "genre_name")

In [0]:

results_group_movie_genre = (
    movies_df.filter(movies_df["year_release_date"] > 2015)
    .join(movies_genres_df,
        movies_df["movie_id"] == movies_genres_df["movie_id"], "inner")
    .join(genres_df,
            movies_genres_df["genre_id"] == genres_df["genre_id"])
).select("budget", "revenue", "year_release_date", "genre_name")

In [0]:
from pyspark.sql.functions import sum, avg, max, min, rank, dense_rank,  desc, col
from pyspark.sql.window import Window

In [0]:
budget_rank = Window.partitionBy("year_release_date").orderBy(desc("total_budget"))
revenue_rank = Window.partitionBy("year_release_date").orderBy(desc("total_revenue"))
budget_revenue_rank = (
    Window.partitionBy("year_release_date")
        .orderBy(
            desc("total_budget"), desc("total_revenue"))
)

results_group_movie_genre = (
    results_group_movie_genre
    .groupBy("year_release_date","genre_name")
        .agg(
            sum("budget").alias("total_budget"),
            sum("revenue").alias("total_revenue")  
        )
    .withColumn("budget_rank", rank().over(budget_rank))
    .withColumn("revenue_dense_rank", dense_rank().over(revenue_rank))
    .withColumn("budget_revenue_rank", rank().over(budget_revenue_rank))
)
display(results_group_movie_genre)

year_release_date,genre_name,total_budget,total_revenue,budget_rank,revenue_dense_rank,budget_revenue_rank
2016,Action,2.5085E9,7.483476635E9,1,2,1
2016,Adventure,2.35E9,7.639395987E9,2,1,2
2016,Fantasy,1.408E9,4.117983742E9,3,3,3
2016,Science Fiction,1.12E9,3.472008555E9,4,4,4
2016,Drama,8.7125E8,2.124354463E9,5,7,5
2016,Comedy,8.655E8,2.986433856E9,6,5,6
2016,Thriller,6.6715E8,1.724250616E9,7,8,7
2016,Family,5.5E8,2.613452629E9,8,6,8
2016,Crime,4.0375E8,1.424057773E9,9,9,9
2016,Horror,1.894E8,8.35561252E8,10,11,10


In [0]:
(
    results_group_movie_genre
    .write
        .mode("overwrite")
        .format("parquet")
        .save(f"{gold_folder_path}/results_group_movie_genre")
)